### Chatbot and RAG Evaluation
##### Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Model (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications. Here, how to evaluate your RAG applications using LangSmith. 

##### 1. How to create datasets?
##### 2. How to run your RAG application on those datasets?
##### 3. How to measure your application's performance using different evaluation metrics?

### Overview:

##### A typical RAG evaluation workflow consists of 3 main steps: 
##### 1. Creating a dataset with questions and their expected answers 
##### 2. Running your RAG application on those questions 
##### 3. Using evaluators to meansure how well your application performed, looking at factors like: 
- Answer relevance 
- Answer accuracy 
- Retrieval quality 

### For this application, we will create and evaluate the bot that answers questions about a few of Lilian Weng's insightful blog posts.


### Chatbot evaluation

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv() 

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [12]:
### STEP1: Create/gather data points
# !pip install langsmith
from langsmith import Client # this client is reponsible to upload the dataset
client=Client()

### Define the dataset - these are your test data
dataset_name="vean_rag_evaluation"
dataset=client.create_dataset(dataset_name)   #create_dataset() method creates a dataset inside langsmith API

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs":{"question":"What is LangChain?"},
            "outputs":{"answer":"A framework building LLM applications"},
        },
        {
            "inputs":{"question":"What is LangSmith?"},
            "outputs":{"answer":"A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs":{"question":"What is OpenAI?"},
            "outputs":{"answer":"A company that creates Large Language Models"},
        },
        {
            "inputs":{"question":"What is Google?"},
            "outputs":{"answer":"A technology company known for search"},
        },
        {
            "inputs":{"question":"What is Mistral?"},
            "outputs":{"answer":"A company that creates Large Language Models"},
        },

    ]
)

# These are ground truths for this application 

{'example_ids': ['4a43e2d3-40e9-4727-8694-68b0dd035c05',
  '512cbe0a-8be3-4314-9aed-2da8bdffee5d',
  '9be2fcc3-9be1-4462-bc3d-c19fd3085885',
  '6d76ab8f-81cc-4f2b-95ab-b9156ac1089a',
  '7c782dc1-fd43-4339-b9df-ec68cc2f8933'],
 'count': 5,
 'as_of': '2026-03-04T22:38:13.612977468Z'}

In [33]:
### STEP2: Define the metrics (LLM as a Judge)
#!pip install openai
import openai 
from langsmith import wrappers # this wrapper module provides convenient tracing wrappers for popular libraries

openai_client=wrappers.wrap_openai(openai.OpenAI()) # wrap_openai patch the openAI client to make it traceable.

eval_instructions="You are an expert professor specialized in grading students' answers to questions"

def correctness(inputs:dict,outputs:dict,reference_outputs:dict)->bool:
    user_content=f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Response with CORRECT or INCORRECT:
    Grade:
    """
    response=openai_client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role":"system","content":eval_instructions},
                {"role":"user","content":user_content}
            ]
        ).choices[0].message.content

    return response=="CORRECT"



In [34]:
### Another metric, Concisions - checks whether the acutal output is less than 2x the length of the expected result
def concision(outputs:dict,reference_outputs:dict)->bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))



In [35]:
### STEP3: RUN Evaluation 
default_instructions="Respond to the user question in a short,concise manner (one short sentence)"

def my_app(question:str,model:str="gpt-4o-mini",instructions:str=default_instructions)->str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role":"system","content":instructions},
            {"role":"user","content":question}
        ]
    ).choices[0].message.content

In [36]:
### call my_app for every data point and compare 
def ls_target(inputs:str)->dict:
    return {"response":my_app(inputs["question"])}

In [37]:
### Run the evaluation
experiment_results=client.evaluate(
    ls_target,  ### Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-4o-mini-chatbot"
)

View the evaluation results for experiment: 'openai-4o-mini-chatbot-6ba1b3d2' at:
https://smith.langchain.com/o/b27cc5db-03bf-4b7a-8622-c1b6fe314bb1/datasets/79280316-e186-4ddf-b90e-7ed3a30d7938/compare?selectedSessions=e4984605-2a18-4c4f-b817-dcbda4f38a2c




0it [00:00, ?it/s]

In [38]:
#try with other model 
### call my_app for every data point and compare 
def ls_target(inputs:str)->dict:
    return {"response":my_app(inputs["question"],model="gpt-4-turbo")}

In [ ]:
### Run the evaluation with turbo 
experiment_results=client.evaluate(
    ls_target,  ### Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-4-turbo-chatbot"
)

View the evaluation results for experiment: 'openai-4-turbo-chatbot-eafffb90' at:
https://smith.langchain.com/o/b27cc5db-03bf-4b7a-8622-c1b6fe314bb1/datasets/79280316-e186-4ddf-b90e-7ed3a30d7938/compare?selectedSessions=14d2236a-d96c-4390-ba0a-4620baf81115




0it [00:00, ?it/s]

### Evaluation of RAG

In [17]:
### RAG pipeline (Document Loaders --> Text splitters --> Embeddings --> Vector Stores)
# ! pip install langchain langchain-community langchain-openai langsmith
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from 
urls=[
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load the documents from the URLs
docs=[WebBaseLoader(url).load() for url in urls]
docs_list=[item for sublist in docs for item in sublist]

# Initialize the text splitter with specified chunk size and overlap 
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,chunk_overlap=0
)
# Split the documents into chunks 
doc_splits=text_splitter.split_documents(docs_list)

# Add the document chunks to "Vector store" using OpenAIEmbeddings
vectorstore=InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=OpenAIEmbeddings(),
)

# With Langchain we can easily turn any vector store into a retrieval component:
retriever=vectorstore.as_retriever(k=6)





USER_AGENT environment variable not set, consider setting it to identify your requests.


In [18]:
# call or test retriever 
retriever.invoke("What is the agent")

[Document(id='75fb3dd7-79e5-4cf7-a727-d25d3fcf949a', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [19]:
# Define LLM 
from langchain.chat_models import init_chat_model 
llm=init_chat_model("openai:gpt-4o-mini")
llm


ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001D6A11EE030>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001D6A11EE250>, root_client=<openai.OpenAI object at 0x000001D6A14BF650>, root_async_client=<openai.AsyncOpenAI object at 0x000001D6A14BF750>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [20]:
from langsmith import traceable

# Add decorator
@traceable
def rag_bot(question: str) -> dict:
    
    # Get relevant context
    docs = retriever.invoke(question)

    docs_string = " ".join(doc.page_content for doc in docs)

    # Create prompt
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.
If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}
"""

    # invoke llm
    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])

    return {"answer": ai_msg.content, "documents": docs}

In [21]:
rag_bot("What is agent")

{'answer': 'An agent, in this context, refers to a system or entity driven by a large language model (LLM) that can perform tasks autonomously by planning, reflecting, and interacting with its environment and other agents. These agents are designed to simulate human-like behavior and decision-making processes. They can also incorporate memory and learning from past experiences to improve their actions over time.',
 'documents': [Document(id='75fb3dd7-79e5-4cf7-a727-d25d3fcf949a', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overvi

### Dataset

In [23]:
from langsmith import Client
client=Client()

examples=[
        {
            "inputs":{"question":"How does the ReAct agent use self-reflection? "},
            "outputs":{"answer":"The ReAct (Reason + Act) agent is designed to combine reasoning and actions while interacting with tools (search APIs, databases, vector stores, etc.)"},
        },
        {
            "inputs":{"question":"What are the types of biases that can arise with few-shot prompting?"},
            "outputs":{"answer":"1.Selection Bias 2.Confirmation Bias 3.Framing/Prompt Bias 4. Anchoring Bias 5. Overgeneralization"},
        },
        {
            "inputs":{"question":"What are the five types of adversial attacks?"},
            "outputs":{"answer":"1. Evasion Attacks 2. Poisioning Attacks 3.Membership Inference Attacks 4. Model Inversion/Privacy Attacks 5. Trojan/Backdoor Attacks"},
        },
        

    ]

### Create the dataset and example in LangSmith 
dataset_name="RAG TEST EVALUATION"
dataset=client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['913392bc-385a-40c5-a1fe-15e9425d93dd',
  '406b6da1-4eb0-4ee0-8b91-636d9312d750',
  '2bb247f7-c10c-4863-bcf0-5e02c77b2169'],
 'count': 3,
 'as_of': '2026-03-05T05:54:34.471754782Z'}

### Evaluators or Metrics
#### Metric#1. Correctness: Response vs reference answer 
- Goal: Measure "How similar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode: Requires a ground truth (reference) answer supplied through a dataset
- Evaluator: Use LLM-as-Judge to assess answer correctness

In [37]:
from typing_extensions import Annotated,TypedDict

## Correctness Output Schema

class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them
    # It is useful to put explanations before responses because it forces the model to think through
    # Its final response before generating it:
    explanation:Annotated[str, ...,"Explain you reasoning for the score"]
    correct:Annotated[bool, ...,"True if the answer is correct, False Otherwise"]

## Correctness prompt 
correctness_instructions = """ You are a teacher grading a quiz.
You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer.
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information thatn the ground truth answer, as long as it is accurate.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

# create a llm 

from langchain_openai import ChatOpenAI
grader_llm=ChatOpenAI(model="gpt-4o-mini",temperature=0).with_structured_output(CorrectnessGrade,method="json_schema",strict=True)

# First evaluator metric - "correctness"
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """
    Evaluates RAG answer correctness using an LLM grader.
    Returns True/False.
    """
    # Construct prompt for grader LLM
    answers = f"""QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}
"""

    # Invoke LLM grader
    grade_response = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])
    return grade["correctness"]

### Metric#2: Relevance: Response vs input 
##### The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance - as in, did the model address the user's question or not.

In [39]:
from typing_extensions import Annotated,TypedDict

## Grade Output Schema (RelevanceGrade)

class RelevanceGrade(TypedDict):
    explanation:Annotated[str,...,"Explain you reasoning for the score"]
    correct:Annotated[bool,...,"Provide the score on weather the answer addresses the question"]

## Grade prompt 
relevance_instructions = """ You are a teacher grading a quiz.
You will be given a QUESTION, a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION


Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
from langchain_openai import ChatOpenAI
relevance_llm=ChatOpenAI(model="gpt-4o",temperature=0).with_structured_output(RelevanceGrade,method="json_schema",strict=True)

# Evaluator
# here comparing inputs and outputs
# inputs given by us & outputs generated by LLM
# Second evaluator metric - "relevance"
def relevance(inputs: dict, outputs: dict) -> bool:
    """
    A simple evaluator for RAG answer helpfulness.
    """
    # Construct prompt for grader LLM
    answer = f"""QUESTION: {inputs['question']}
    STUDENT ANSWER: {outputs['answer']}"""


    # Invoke LLM grader
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Metric#3:Groundedness: Response vs retrieved docs 
#### Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [40]:
from typing_extensions import Annotated,TypedDict

## Grade Output Schema (RelevanceGrade)

class GroundedGrade(TypedDict):
    explanation:Annotated[str,...,"Explain you reasoning for the score"]
    grounded:Annotated[bool,...,"Provide the score on if the answer hallucinates from the documents"]

## Grade prompt 
grounded_instructions = """ You are a teacher grading a quiz.
You will be given FACTS and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the score of FACTS


Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
from langchain_openai import ChatOpenAI
grounded_llm=ChatOpenAI(model="gpt-4o",temperature=0).with_structured_output(GroundedGrade,method="json_schema",strict=True)

# Evaluator
# here comparing inputs and outputs
# inputs given by us & outputs generated by LLM

def groundedness(inputs: dict, outputs: dict) -> bool:
    """
    A simple evaluator for RAG answer groundedness.
    """
    # Construct prompt for grader LLM
    doc_string="\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS:{doc_string}\nSTUDENT ANSWER:{outputs['answer']}"
    

    # Invoke LLM grader
    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["grounded"]

### Metric #4: Retrieval Relevance: Retrieved docs vs input 

In [45]:
from typing_extensions import Annotated,TypedDict

## Grade Output Schema (RelevanceGrade)

class RetrievalRelevanceGrade(TypedDict):
    explanation:Annotated[str,...,"Explain you reasoning for the score"]
    relevant:Annotated[bool,...,"True if the retrieved documents are relevant to the question, False if not relevant"]

## Grade prompt 
retrieval_relevance_instructions = """ You are a teacher grading a quiz.
You will be given QUESTION and a set of FACTS provided by the student.

Here is the grade criteria to follow:
(1) Your goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant.
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is measurable


Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the question
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
from langchain_openai import ChatOpenAI
retrieval_relevance_llm=ChatOpenAI(model="gpt-4o",temperature=0).with_structured_output(RetrievalRelevanceGrade,method="json_schema",strict=True)

# Evaluator
# here comparing inputs and outputs
# inputs given by us & outputs generated by LLM

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """
    A simple evaluator for document relevance.
    """
    # Construct prompt for grader LLM
    docs = outputs.get("documents", [])
    doc_string="\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS:{doc_string}\nQUESTION:{inputs['question']}"
    

    # Invoke LLM grader
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Run the evaluation 

In [ ]:
# Run the evaluation with a separate function
def target(inputs:dict)-> dict:
    return rag_bot(inputs["question"])

experiment_results=client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness,groundedness,relevance,retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version":"LECL context,gpt-d-0125-preview"},
)

In [34]:
# Explore results locally as dataframe if you have pandas installed
experiment_results.to_pandas()

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.wrapper,feedback.groundedness,feedback.retrieval_relevance,execution_time,example_id,id
0,What are the five types of adversial attacks?,The five types of adversarial attacks mentione...,[page_content='Black-box attacks assume that a...,None,1. Evasion Attacks 2. Poisioning Attacks 3.Mem...,None,True,True,5.227707,2bb247f7-c10c-4863-bcf0-5e02c77b2169,019cbc92-2bce-73c0-b0b2-8563c80162c8
1,What are the types of biases that can arise wi...,The types of biases that can arise with few-sh...,[page_content='Zero-shot and few-shot learning...,None,1.Selection Bias 2.Confirmation Bias 3.Framing...,None,True,True,2.803496,406b6da1-4eb0-4ee0-8b91-636d9312d750,019cbc92-6613-7fa1-ac39-e923d3d59824
2,How does the ReAct agent use self-reflection?,The ReAct agent uses self-reflection by incorp...,[page_content='Self-reflection is a vital aspe...,None,The ReAct (Reason + Act) agent is designed to ...,None,False,False,2.600075,913392bc-385a-40c5-a1fe-15e9425d93dd,019cbc92-96e7-72e1-a797-050a9dd65df6
